In [51]:
# Imports + Funções utilitárias

from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as spark_sum
import time

In [52]:
# SparkSession

spark = (
    SparkSession.builder
    .appName("montee_en_charge_taxi")
    .master("spark://172.22.114.87:7077")
    .getOrCreate()
)

spark

In [62]:
# Sanity check

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)

Spark version: 3.5.1
Master: spark://172.22.114.87:7077


In [63]:
# Lecture 12 Parquets

paths = [f"/home/user/bigdata/taxi/yellow_tripdata_2023-{m:02d}.parquet" for m in range(1,13)]

dfs = [spark.read.parquet(p) for p in paths]

df_1m = dfs[0]  # janvier

In [55]:
import os

for p in paths:
    print(p, os.path.exists(p))

/home/user/bigdata/taxi/yellow_tripdata_2023-01.parquet True
/home/user/bigdata/taxi/yellow_tripdata_2023-02.parquet True
/home/user/bigdata/taxi/yellow_tripdata_2023-03.parquet True
/home/user/bigdata/taxi/yellow_tripdata_2023-04.parquet True
/home/user/bigdata/taxi/yellow_tripdata_2023-05.parquet True
/home/user/bigdata/taxi/yellow_tripdata_2023-06.parquet True
/home/user/bigdata/taxi/yellow_tripdata_2023-07.parquet True
/home/user/bigdata/taxi/yellow_tripdata_2023-08.parquet True
/home/user/bigdata/taxi/yellow_tripdata_2023-09.parquet True
/home/user/bigdata/taxi/yellow_tripdata_2023-10.parquet True
/home/user/bigdata/taxi/yellow_tripdata_2023-11.parquet True
/home/user/bigdata/taxi/yellow_tripdata_2023-12.parquet True


In [64]:
# Datasets 3m / 6m / 12m

# 3 mois : janvier–mars
df_3m = dfs[0].union(dfs[1]).union(dfs[2])

# 6 mois : janvier–juin
df_6m = df_3m.union(dfs[3]).union(dfs[4]).union(dfs[5])

# 12 mois : janvier–décembre
df_12m = dfs[0]
for d in dfs[1:]:
    df_12m = df_12m.union(d)

In [66]:
# Schema et top 5

df_1m.printSchema()
df_1m.show(5)

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+----

In [67]:
# Fonctions benchmark

def benchmark_count(df, label):
    t0 = time.time()
    n = df.count()
    t1 = time.time()
    dt = t1 - t0
    print(f"{label}: {n} lignes, temps = {dt:.2f} sec")
    return n, dt

def benchmark_reduce(df, label):
    t0 = time.time()
    total = df.select(spark_sum("fare_amount")).collect()[0][0]
    t1 = time.time()
    dt = t1 - t0
    print(f"{label}: somme = {total:.2f}, temps = {dt:.2f} sec")
    return float(total), dt

In [68]:
# Execution benchmarks et resultats

results = {}

print("=== COUNT() ===")
results["1m_count"]  = benchmark_count(df_1m,  "1 mois")
results["3m_count"]  = benchmark_count(df_3m,  "3 mois")
results["6m_count"]  = benchmark_count(df_6m,  "6 mois")
results["12m_count"] = benchmark_count(df_12m, "12 mois")

print("\n=== REDUCE (sum fare_amount) ===")
results["1m_reduce"]  = benchmark_reduce(df_1m,  "1 mois")
results["3m_reduce"]  = benchmark_reduce(df_3m,  "3 mois")
results["6m_reduce"]  = benchmark_reduce(df_6m,  "6 mois")
results["12m_reduce"] = benchmark_reduce(df_12m, "12 mois")

results

=== COUNT() ===
1 mois: 3066766 lignes, temps = 0.08 sec
3 mois: 9384487 lignes, temps = 0.12 sec
6 mois: 19493620 lignes, temps = 0.26 sec
12 mois: 38310226 lignes, temps = 0.39 sec

=== REDUCE (sum fare_amount) ===
1 mois: somme = 56327501.54, temps = 0.11 sec
3 mois: somme = 173780804.24, temps = 0.27 sec
6 mois: somme = 373388695.05, temps = 0.28 sec
12 mois: somme = 747901833.91, temps = 0.49 sec


{'1m_count': (3066766, 0.07949471473693848),
 '3m_count': (9384487, 0.1237192153930664),
 '6m_count': (19493620, 0.25974035263061523),
 '12m_count': (38310226, 0.3948202133178711),
 '1m_reduce': (56327501.53999907, 0.11396026611328125),
 '3m_reduce': (173780804.24001175, 0.2666776180267334),
 '6m_reduce': (373388695.050012, 0.28479790687561035),
 '12m_reduce': (747901833.910037, 0.4944314956665039)}

In [69]:
# Tableau montée en charge

import pandas as pd

data = []

labels = [
    ("1 mois",  "1m_count",  "1m_reduce"),
    ("3 mois",  "3m_count",  "3m_reduce"),
    ("6 mois",  "6m_count",  "6m_reduce"),
    ("12 mois", "12m_count", "12m_reduce"),
]

for label, key_count, key_reduce in labels:
    n_rows, t_count = results[key_count]
    total_fare, t_reduce = results[key_reduce]
    data.append({
        "Volume": label,
        "Nombre de lignes": n_rows,
        "Temps COUNT (s)": round(t_count, 2),
        "Temps REDUCE (s)": round(t_reduce, 2),
        "Somme fare_amount": round(total_fare, 2),
    })

df_results = pd.DataFrame(data)
df_results

,Volume,Nombre de lignes,Temps COUNT (s),Temps REDUCE (s),Somme fare_amount
0,1 mois,3066766,0.08,0.11,5.632750e+07
1,3 mois,9384487,0.12,0.27,1.737808e+08
2,6 mois,19493620,0.26,0.28,3.733887e+08
3,12 mois,38310226,0.39,0.49,7.479018e+08


In [70]:
# Exportation en CSV

df_results.to_csv("montee_en_charge_taxi_2023.csv", index=False)
df_results

,Volume,Nombre de lignes,Temps COUNT (s),Temps REDUCE (s),Somme fare_amount
0,1 mois,3066766,0.08,0.11,5.632750e+07
1,3 mois,9384487,0.12,0.27,1.737808e+08
2,6 mois,19493620,0.26,0.28,3.733887e+08
3,12 mois,38310226,0.39,0.49,7.479018e+08
